## Problem

You defined a multimodal pipeline in Python and now need to inspect tables, debug computed columns, roll back changes, and expose HTTP endpoints without writing more application code. Jumping into a REPL or building a custom admin UI for every project does not scale, especially when AI agents need stable, machine-readable output.

## Solution

**What's in this recipe:**

- Inspect catalogs with `pxt ls`, `describe`, `columns`, and `idxs`
- Query and debug rows with `pxt rows`, `count`, `get`, and `errors`
- Manage versions with `pxt history` and `pxt revert`
- Script and automate with `--json`, `-f`, and `pxt shell`
- Validate declarative HTTP serving with `pxt serve --dry-run`

The `pxt` CLI ships with Pixeltable (v0.6.5+). Catalog commands talk to a local daemon (~40 ms per call after the first invocation). Use Python to define schema once, then operate the catalog from the terminal.

See the [CLI reference](https://docs.pixeltable.com/platform/cli) for every flag.

### Setup

In [1]:
%pip install -qU 'pixeltable[serve]'

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import pixeltable as pxt
import subprocess
import sys
from pathlib import Path
from pixeltable.functions.video import frame_iterator


def run_pxt(*args: str) -> subprocess.CompletedProcess[str]:
    """Run a pxt subcommand and print stdout. Raises on non-zero exit."""
    cmd = ['pxt', *args]
    result = subprocess.run(
        cmd, capture_output=True, text=True, check=False
    )
    if result.stdout:
        sys.stdout.write(result.stdout)
    if result.stderr:
        sys.stderr.write(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f'pxt failed ({result.returncode}): {" ".join(cmd)}'
        )
    return result

In [3]:
SAMPLE_VIDEO = 'https://raw.githubusercontent.com/pixeltable/pixeltable/release/docs/resources/bangkok.mp4'

pxt.drop_dir('cli_demo', force=True)
pxt.create_dir('cli_demo')

videos = pxt.create_table(
    'cli_demo/videos', {'video': pxt.Video, 'title': pxt.String}
)
frames = pxt.create_view(
    'cli_demo/frames',
    videos,
    iterator=frame_iterator(videos.video, fps=1),
)
frames.add_computed_column(thumb=frames.frame.thumbnail((320, 180)))

videos.insert([{'video': SAMPLE_VIDEO, 'title': 'Bangkok'}])

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/private/var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home/pgdata
Created directory 'cli_demo'.
Created table 'videos'.
Added 0 column values with 0 errors in 0.00 s


Inserted 20 rows with 0 errors in 0.98 s (20.48 rows/s)


20 rows inserted.

### 1. Inspect the catalog

List directories and tables, then drill into schema and computed columns. Flag letters in `pxt ls -l`: `c` = computed column, `i` = index.

In [4]:
run_pxt('ls', '-l', 'cli_demo')
print('---')
run_pxt('describe', 'cli_demo/videos')
print('---')
run_pxt('columns', 'cli_demo/frames', '--computed')
print('---')
run_pxt('idxs', 'cli_demo/frames')

path             kind   cols  version  flags
cli_demo/frames  view      6        2  c
cli_demo/videos  table     2        1  i
---
table 'cli_demo/videos'

 Column Name    Type  Source Computed With Comment
--------------------------------------------------
       video   Video  videos                      
       title  String  videos                      
---


cli_demo/frames	thumb	Required[Image]	computed	frame.thumbnail([320, 180])
---


CompletedProcess(args=['pxt', 'idxs', 'cli_demo/frames'], returncode=0, stdout='', stderr='')

### 2. Query rows

Peek at stored data from the terminal. Pass computed columns explicitly with `--cols`; unstored computed columns are skipped by default. Thumbnails may take a moment to compute after insert.

In [5]:
run_pxt('count', 'cli_demo/frames')
print('---')
run_pxt('rows', 'cli_demo/frames', '-n', '1', '--cols', 'pos,thumb')

19
---


pos	thumb
0	<Image 320x180 RGB>


CompletedProcess(args=['pxt', 'rows', 'cli_demo/frames', '-n', '1', '--cols', 'pos,thumb'], returncode=0, stdout='pos\tthumb\n0\t<Image 320x180 RGB>\n', stderr='')

### 3. Debug computed-column failures

When a stored computed column fails, `pxt errors` lists the failing rows by primary key.

In [6]:
@pxt.udf
def boom_if_zero(x: int) -> int:
    if x == 0:
        raise ValueError('boom')
    return x


failures = pxt.create_table(
    'cli_demo/failures', {'k': pxt.Required[pxt.Int]}, primary_key='k'
)
failures.add_computed_column(
    result=boom_if_zero(failures.k), on_error='ignore'
)
failures.insert([{'k': 0}, {'k': 1}], on_error='ignore')

run_pxt('errors', 'cli_demo/failures')

Created table 'failures'.
Added 0 column values with 0 errors in 0.01 s
Inserted 2 rows with 2 errors across 2 columns (failures.result, failures.None) in 0.01 s (393.29 rows/s)


{k: 0}	result	ValueError	boom


CompletedProcess(args=['pxt', 'errors', 'cli_demo/failures'], returncode=0, stdout='{k: 0}\tresult\tValueError\tboom\n', stderr='')

### 4. Version control

Every insert and schema change creates a new table version. Inspect the timeline, then roll back if needed. See [Track changes and revert](/howto/cookbooks/core/version-control-history) for the Python API.

In [7]:
videos.add_computed_column(label=videos.title.upper())

run_pxt('history', 'cli_demo/videos', '-n', '5')
print('---')
run_pxt('revert', 'cli_demo/videos', '-f')
print('---')
run_pxt('history', 'cli_demo/videos', '-n', '3')

Added 1 column value with 0 errors in 0.02 s (55.71 rows/s)


version	created_at	change_type	inserts	updates	deletes	errors	schema_change
2	2026-06-12T23:31:02.604384Z	schema	0	1	0	0	Added: label
1	2026-06-12T23:31:00.982535Z	data	20	0	0	0	
0	2026-06-12T23:31:00.684313Z	schema	0	0	0	0	Initial Version
---
reverted cli_demo/videos: v2 -> v1
---


version	created_at	change_type	inserts	updates	deletes	errors	schema_change
1	2026-06-12T23:31:00.982535Z	data	20	0	0	0	
0	2026-06-12T23:31:00.684313Z	schema	0	0	0	0	Initial Version


CompletedProcess(args=['pxt', 'history', 'cli_demo/videos', '-n', '3'], returncode=0, stdout='version\tcreated_at\tchange_type\tinserts\tupdates\tdeletes\terrors\tschema_change\n1\t2026-06-12T23:31:00.982535Z\tdata\t20\t0\t0\t0\t\n0\t2026-06-12T23:31:00.684313Z\tschema\t0\t0\t0\t0\tInitial Version\n', stderr='')

### 5. Agent-friendly scripting

Most catalog commands accept `--json` for stable, machine-readable output. Use `-f` to skip confirmation prompts in non-interactive contexts.

For many commands in one session, `pxt shell` keeps the daemon warm:

```bash
pxt shell
pxt> ls cli_demo
pxt> count cli_demo/frames
pxt> exit
```

In [8]:
ls_json = json.loads(run_pxt('ls', 'cli_demo', '--json').stdout)
table_paths = [
    e['path'] for e in ls_json['entries'] if e['kind'] == 'table'
]
print('tables:', table_paths)

count_json = json.loads(
    run_pxt('count', 'cli_demo/frames', '--json').stdout
)
print('frame count:', count_json['count'])

{
  "entries": [
    {
      "path": "cli_demo/failures",
      "kind": "table",
      "num_rows": null,
      "num_cols": null,
      "last_version": 2,
      "flags": ""
    },
    {
      "path": "cli_demo/frames",
      "kind": "view",
      "num_rows": null,
      "num_cols": null,
      "last_version": 1,
      "flags": ""
    },
    {
      "path": "cli_demo/videos",
      "kind": "table",
      "num_rows": null,
      "num_cols": null,
      "last_version": 1,
      "flags": ""
    }
  ],
  "tree": null
}
tables: ['cli_demo/failures', 'cli_demo/videos']
{
  "path": "cli_demo/frames",
  "count": 0
}
frame count: 0


### 6. Config and health

Check daemon health, runtime status, and resolved configuration (API keys show as `<redacted>` when set). See [Configure API keys](/howto/cookbooks/core/workflow-api-keys) for credential setup.

In [9]:
health = json.loads(run_pxt('health').stdout)
print('daemon ok:', health['ok'], '| pxt version:', health['pxt_version'])
print('---')
run_pxt('status')
print('---')
run_pxt('config', '--section', 'openai')

{
  "ok": true,
  "service": "pxt",
  "pxt_version": "0.6.5",
  "pid": 52061,
  "started_at": "2026-06-12T23:09:23.075920+00:00",
  "pxt_install_dir": "/opt/miniconda3/envs/pxt/lib/python3.10/site-packages/pixeltable",
  "python_executable": "/opt/miniconda3/envs/pxt/bin/python",
  "pixeltable_home": "/private/var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home",
  "pixeltable_pgdata": "/private/var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home/pgdata",
  "pixeltable_config_file": "/private/var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home/config.toml",
  "pixeltable_env": {
    "PIXELTABLE_HOME": "/var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home"
  }
}
daemon ok: True | pxt version: 0.6.5
---


pxt_version     0.6.5.dev24+8a39208c
daemon_pid      52061
daemon_started  2026-06-12T23:09:23.075920+00:00
home            /var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home
db_url          postgresql+psycopg://postgres:***@/pixeltable?host=%2Fprivate%2Fvar%2Ffolders%2Fs4%2F0zdx499s6sv3_0jll6ccdbh00000gn%2FT%2Ftmp.8rYgq6oxHN%2F.pxt-home%2Fpgdata
media_dir       /var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home/media
file_cache_dir  /var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home/file_cache
total_tables    7
total_errors    2
---
config_file  /var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home/config.toml
not set: openai.api_key, openai.base_url, openai.api_version, openai.rate_limits, openai.max_connections, openai.max_keepalive_connections, openai.read_timeout, openai.write_timeout


CompletedProcess(args=['pxt', 'config', '--section', 'openai'], returncode=0, stdout='config_file  /var/folders/s4/0zdx499s6sv3_0jll6ccdbh00000gn/T/tmp.8rYgq6oxHN/.pxt-home/config.toml\nnot set: openai.api_key, openai.base_url, openai.api_version, openai.rate_limits, openai.max_connections, openai.max_keepalive_connections, openai.read_timeout, openai.write_timeout\n', stderr='')

### 7. Serve without application code

Declare HTTP routes in `pyproject.toml` and validate the config with `--dry-run --json` (no server started). For the full live flow:

```bash
python demo.py                        # create tables (already done in this notebook)
pxt serve video-api                   # start REST API from pyproject.toml
curl -X POST localhost:8000/videos \
  -H 'Content-Type: application/json' \
  -d '{"video": "https://raw.githubusercontent.com/pixeltable/pixeltable/release/docs/resources/bangkok.mp4", "title": "Bangkok"}'
pxt rows cli_demo/frames -n 1 --cols pos,thumb
```

See [HTTP Serving](https://docs.pixeltable.com/howto/deployment/serving) for the TOML reference.

In [10]:
serve_dir = Path('_cli_demo_serve')
serve_dir.mkdir(exist_ok=True)
(serve_dir / 'demo.py').write_text(
    '# optional; tables already created above\n'
)
(serve_dir / 'pyproject.toml').write_text(
    """[project]
name = "cli-demo"
version = "0.1.0"

[[tool.pixeltable.service]]
name = "video-api"

[[tool.pixeltable.service.routes]]
type = "insert"
path = "/videos"
table = "cli_demo/videos"
inputs = ["video", "title"]
outputs = ["title"]
"""
)

result = subprocess.run(
    ['pxt', 'serve', 'video-api', '--dry-run', '--json'],
    cwd=serve_dir,
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(result.stderr)

{
  "name": "video-api",
  "prefix": "",
  "host": "0.0.0.0",
  "port": 8000,
  "routes": [
    {
      "path": "/videos",
      "background": false,
      "type": "insert",
      "table": "cli_demo/videos",
      "inputs": [
        "video",
        "title"
      ],
      "uploadfile_inputs": null,
      "outputs": [
        "title"
      ],
      "return_fileresponse": false,
      "export_sql": null
    }
  ]
}



### Next steps

- [CLI reference](https://docs.pixeltable.com/platform/cli): every command and flag
- [Dashboard](https://docs.pixeltable.com/platform/dashboard): browse tables and preview media in the browser
- [HTTP Serving](https://docs.pixeltable.com/howto/deployment/serving): production TOML and `FastAPIRouter`
- [Building with LLMs](https://docs.pixeltable.com/overview/building-pixeltable-with-llms): agent skills, MCP, and `pxt --json` workflows